# NYC Airbnb EDA

This notebook fetches the `sample.csv` artifact from Weights & Biases, inspects the dataset, profiles it, and reproduces the initial cleaning decisions used later in the ML pipeline.

In [ ]:
import pandas as pd
import wandb
from ydata_profiling import ProfileReport

In [ ]:
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
artifact = run.use_artifact("sample.csv:latest")
local_path = artifact.file(root="artifacts/sample.csv_latest")
df = pd.read_csv(local_path)
df.head()

## Initial inspection

Check the schema, missing values, duplicated rows, and price distribution before deciding on deterministic cleaning rules.

In [ ]:
df.info()
df.isna().sum().sort_values(ascending=False)
df.duplicated().sum()

In [ ]:
df['price'].describe()

In [ ]:
profile = ProfileReport(df, title="NYC Airbnb Sample Profile", explorative=True)
profile.to_widgets()

## Cleaning decisions

The pipeline keeps missing values for later imputation, but applies deterministic cleaning here: filter unrealistic nightly prices and convert `last_review` to datetime.

In [ ]:
min_price = 10
max_price = 350
price_mask = df['price'].between(min_price, max_price)
df_clean = df.loc[price_mask].copy()
df_clean['last_review'] = pd.to_datetime(df_clean['last_review'])
df_clean.head()

In [ ]:
df_clean.info()
df_clean['price'].describe()

In [ ]:
run.finish()